Connect to the drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Installs

In [2]:
!pip install -q transformers datasets accelerate evaluate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00


Imports

In [3]:
import os
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer
)

from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

GPU or CPU

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device cuda
GPU: NVIDIA A100-SXM4-80GB


Daroberta Bro

In [6]:
MODEL_NAME = "microsoft/deberta-v3-large"

MAX_LENGTH = 256
NUM_LABELS = 3 # Valence, Arousal, Dominance

Data

In [7]:
emobank_path = "/content/drive/MyDrive/Thesis docs/emobank.csv"

emo_df = pd.read_csv(emobank_path)

emo_df.head()

,id,split,V,A,D,text
0,110CYL068_1036_1079,train,3.00,3.00,3.20,"Remember what she said in my last letter? """
1,110CYL068_1079_1110,test,2.80,3.10,2.80,If I wasn't working here.
2,110CYL068_1127_1130,train,3.00,3.00,3.00,".."""
3,110CYL068_1137_1188,train,3.44,3.00,3.22,Goodwill helps people get off of public assist...
4,110CYL068_1189_1328,train,3.55,3.27,3.46,Sherry learned through our Future Works class ...


In [8]:
emo_df.shape

(10062, 6)

In [9]:
emo_df.columns

Index(['id', 'split', 'V', 'A', 'D', 'text'], dtype='object')

In [10]:
emo_df = emo_df.rename(columns={
    "V": "valence",
    "A": "arousal",
    "D": "dominance",
})

emo_df = emo_df[["text", "valence", "arousal", "dominance", "split"]].dropna()

emo_df.head()

,text,valence,arousal,dominance,split
0,"Remember what she said in my last letter? """,3.00,3.00,3.20,train
1,If I wasn't working here.,2.80,3.10,2.80,test
2,"..""",3.00,3.00,3.00,train
3,Goodwill helps people get off of public assist...,3.44,3.00,3.22,train
4,Sherry learned through our Future Works class ...,3.55,3.27,3.46,train


In [11]:
emo_df.shape

(10061, 5)

In [12]:
emo_df.columns

Index(['text', 'valence', 'arousal', 'dominance', 'split'], dtype='object')

In [13]:
train_df = emo_df[emo_df["split"] == "train"].copy()
val_df = emo_df[emo_df["split"] == "dev"].copy()
test_df = emo_df[emo_df["split"] == "test"].copy()

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (8062, 5)
Validation shape: (999, 5)
Test shape: (1000, 5)


Hugging Face datasets

In [14]:
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index()),
    "validation": Dataset.from_pandas(val_df.reset_index()),
    "test": Dataset.from_pandas(test_df.reset_index()),
})

dataset

DatasetDict({
    train: Dataset({
        features: ['index', 'text', 'valence', 'arousal', 'dominance', 'split'],
        num_rows: 8062
    })
    validation: Dataset({
        features: ['index', 'text', 'valence', 'arousal', 'dominance', 'split'],
        num_rows: 999
    })
    test: Dataset({
        features: ['index', 'text', 'valence', 'arousal', 'dominance', 'split'],
        num_rows: 1000
    })
})

Start the tokenizer

In [15]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Prepare the data

In [16]:
def preprocess(batch):
    tokenized = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

    tokenized["labels"] = [
        [v, a, d] 
        for v, a, d in zip(
            batch["valence"], 
            batch["arousal"], 
            batch["dominance"]
        )
    ]

    return tokenized

tokenized_dataset = dataset.map(
    preprocess,
    batched=True,
    remove_columns=dataset["train"].column_names
)

tokenized_dataset

Map:   0%|          | 0/8062 [00:00<?, ? examples/s]

Map:   0%|          | 0/999 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 8062
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 999
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1000
    })
})

A tiny check

In [17]:
tokenized_dataset["train"][0].keys()

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])

Load the model

In [29]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="regression",
    dtype=torch.float32
)

model.to(device)

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifie

DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 1024, padding_idx=0)
      (LayerNorm): LayerNorm((1024,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-23): 24 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (key_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (value_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNo

In [32]:
model.config

DebertaV2Config {
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  "layer_norm_eps": 1e-07,
  "legacy": true,
  "max_position_embeddings": 512,
  "max_relative_positions": -1,
  "model_type": "deberta-v2",
  "norm_rel_ebd": "layer_norm",
  "num_attention_heads": 16,
  "num_hidden_layers": 24,
  "pad_token_id": 0,
  "pooler_dropout": 0.0,
  "pooler_hidden_act": "gelu",
  "pooler_hidden_size": 1024,
  "pos_att_type": [
    "p2c",
    "c2p"
  ],
  "position_biased_input": false,
  "position_buckets": 256,
  "problem_type": "regression",
  "relative_attention": true,
  "share_att_key": true,
  "tie_word_embeddings": true,
  "transformers_version

In [48]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # For regression, predictions are already continuous values :)
    preds = predictions

    metrics = {}

    target_names = ['valence', 'arousal', 'dominance']

    for i, name in enumerate(target_names):
        y_true = labels[:, i]
        y_pred = preds[:, i]

        metrics[f"{name}_pearson"] = pearsonr(y_true, y_pred)[0]
        mse = mean_squared_error(y_true, y_pred)
        metrics[f"{name}_rmse"] = float(np.sqrt(mse))
        metrics[f"{name}_mae"] = mean_absolute_error(y_true, y_pred)

    metrics["mean_pearson"] = np.mean([
        metrics["valence_pearson"],
        metrics["arousal_pearson"],
        metrics["dominance_pearson"]
    ])

    metrics["mean_rmse"] = np.mean([
        metrics["valence_rmse"],
        metrics["arousal_rmse"],
        metrics["dominance_rmse"]
    ])

    metrics["mean_mae"] = np.mean([
        metrics["valence_mae"],
        metrics["arousal_mae"],
        metrics["dominance_mae"]
    ])

    return metrics

In [65]:
OUT_DIR = "/content/drive/MyDrive/Thesis docs/models/Filipa-model"
LOG_DIR = "/content/drive/MyDrive/Thesis docs/logs/filipa-model"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print("Model will be saved to:", OUT_DIR)

Model will be saved to: /content/drive/MyDrive/Thesis docs/models/Filipa-model


Reproducibility

In [50]:
from transformers import set_seed, EarlyStoppingCallback

SEED = 42
set_seed(SEED)

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Training args

In [56]:
training_args = TrainingArguments(
    output_dir=OUT_DIR,

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=5e-6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,

    num_train_epochs=15,
    warmup_steps=0.06,
    weight_decay=0.01,

    lr_scheduler_type="linear",

    fp16=torch.cuda.is_available(),

    load_best_model_at_end=True,
    metric_for_best_model="mean_pearson",
    greater_is_better=True,

    logging_dir=LOG_DIR,
    logging_steps=50,

    save_total_limit=2,
    report_to="none",
    seed=SEED
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Trainer

In [57]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=4,
        )
    ]
)

DANGER train

In [58]:
train_output = trainer.train()

train_output

Epoch,Training Loss,Validation Loss,Valence Pearson,Valence Rmse,Valence Mae,Arousal Pearson,Arousal Rmse,Arousal Mae,Dominance Pearson,Dominance Rmse,Dominance Mae,Mean Pearson,Mean Rmse,Mean Mae
1,0.068592,0.039612,0.839899,0.195420,0.148826,0.621085,0.209837,0.165253,0.521276,0.191352,0.150378,0.660753,0.198870,0.154819
2,0.060343,0.044272,0.842306,0.210641,0.164510,0.607264,0.223045,0.174052,0.494749,0.196715,0.151851,0.648106,0.210134,0.163471
3,0.054621,0.052719,0.839741,0.227543,0.180346,0.604544,0.230841,0.181937,0.493277,0.230418,0.184206,0.645854,0.229600,0.182163
4,0.052444,0.043772,0.838566,0.204505,0.156771,0.604058,0.221463,0.174064,0.499955,0.201114,0.158202,0.647526,0.209027,0.163012
5,0.046530,0.049227,0.833282,0.218634,0.166988,0.592257,0.235334,0.183938,0.496806,0.210946,0.166322,0.640782,0.221638,0.172416


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1260, training_loss=0.05522754242022832, metrics={'train_runtime': 753.1044, 'train_samples_per_second': 160.575, 'train_steps_per_second': 5.019, 'total_flos': 1.878326689623552e+16, 'train_loss': 0.05522754242022832, 'epoch': 5.0})

Save model and tokenizer

In [59]:
eval_metrics = trainer.evaluate(tokenized_dataset["validation"])
eval_metrics

Training Loss,Validation Loss,Epoch,Valence Pearson,Valence Rmse,Valence Mae,Arousal Pearson,Arousal Rmse,Arousal Mae,Dominance Pearson,Dominance Rmse,Dominance Mae,Mean Pearson,Mean Rmse,Mean Mae
0.046530,0.039612,5,0.839899,0.195420,0.148826,0.621085,0.209837,0.165253,0.521276,0.191352,0.150378,0.660753,0.198870,0.154819


{'eval_loss': 0.03961212933063507,
 'eval_valence_pearson': 0.8398988842964172,
 'eval_valence_rmse': 0.19542021642102952,
 'eval_valence_mae': 0.1488260179758072,
 'eval_arousal_pearson': 0.6210846304893494,
 'eval_arousal_rmse': 0.20983734041538443,
 'eval_arousal_mae': 0.16525250673294067,
 'eval_dominance_pearson': 0.5212757587432861,
 'eval_dominance_rmse': 0.19135207752533354,
 'eval_dominance_mae': 0.1503780335187912,
 'eval_mean_pearson': 0.660753071308136,
 'eval_mean_rmse': 0.19886987812058252,
 'eval_mean_mae': 0.15481885274251303}

In [67]:
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

print("Saved model to:", OUT_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model to: /content/drive/MyDrive/Thesis docs/models/Filipa-model


In [61]:
history_df = pd.DataFrame(trainer.state.log_history)

history_path = f"{OUT_DIR}/training_history.csv"
history_df.to_csv(history_path, index=False)

history_df.tail()

,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_valence_pearson,eval_valence_rmse,eval_valence_mae,eval_arousal_pearson,...,eval_mean_rmse,eval_mean_mae,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
27,0.045028,1.438121,0.000004,4.761905,1200,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28,0.046530,0.963325,0.000004,4.960317,1250,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29,NaN,NaN,NaN,5.000000,1260,0.049227,0.833282,0.218634,0.166988,0.592257,...,0.221638,0.172416,4.4633,223.823,14.115,NaN,NaN,NaN,NaN,NaN
30,NaN,NaN,NaN,5.000000,1260,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,753.1044,160.575,5.019,1.878327e+16,0.055228
31,NaN,NaN,NaN,5.000000,1260,0.039612,0.839899,0.195420,0.148826,0.621085,...,0.198870,0.154819,5.2061,191.889,12.101,NaN,NaN,NaN,NaN,NaN


In [62]:
test_results = trainer.evaluate(tokenized_dataset["test"])
test_results

Training Loss,Validation Loss,Epoch,Valence Pearson,Valence Rmse,Valence Mae,Arousal Pearson,Arousal Rmse,Arousal Mae,Dominance Pearson,Dominance Rmse,Dominance Mae,Mean Pearson,Mean Rmse,Mean Mae
0.046530,0.041219,5,0.843315,0.194015,0.148101,0.570825,0.218027,0.168522,0.531716,0.196165,0.154295,0.648619,0.202736,0.156973


{'eval_loss': 0.041219428181648254,
 'eval_valence_pearson': 0.8433153629302979,
 'eval_valence_rmse': 0.1940151711122448,
 'eval_valence_mae': 0.1481013149023056,
 'eval_arousal_pearson': 0.5708245635032654,
 'eval_arousal_rmse': 0.2180271979555912,
 'eval_arousal_mae': 0.16852203011512756,
 'eval_dominance_pearson': 0.5317161679267883,
 'eval_dominance_rmse': 0.1961645898867713,
 'eval_dominance_mae': 0.1542953997850418,
 'eval_mean_pearson': 0.6486186981201172,
 'eval_mean_rmse': 0.2027356529848691,
 'eval_mean_mae': 0.15697291493415833}